In [1]:
import sys, os
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df, convert_col
import pickle

Enabled rmm statistics


In [2]:
%load_ext cudf.pandas

In [3]:
%LoadCheckpoint /scratch/jieq/pandax/dias_notebooks/dataranch/supermarket-sales-prediction-xgboost-fastai/src/small_bench/checkpoints/post_cell_7.pickle

trying: ['REGRESSOR', 'SEP_COMMA', 'ENABLE_BREAKPOINT', 'SHUFFLE_DATA']
me:  2
me:  2
me:  2
me:  2
trying: ['REGRESSOR', 'SEP_COMMA', 'ENABLE_BREAKPOINT', 'SHUFFLE_DATA']
me:  2
me:  2
me:  2
me:  2
trying: ['CONVERT_TO_CAT', 'AUTO_ADJUST_LEARNING_RATE', 'VARIABLE_FILES', 'SEP_DOLLAR']
me:  2
me:  2
me:  2
me:  2
trying: ['CONVERT_TO_CAT', 'AUTO_ADJUST_LEARNING_RATE', 'VARIABLE_FILES', 'SEP_DOLLAR']
me:  2
me:  2
me:  2
me:  2
trying: ['REGRESSOR', 'SEP_COMMA', 'ENABLE_BREAKPOINT', 'SHUFFLE_DATA']
me:  2
me:  2
me:  2
me:  2
trying: ['REGRESSOR', 'SEP_COMMA', 'ENABLE_BREAKPOINT', 'SHUFFLE_DATA']
me:  2
me:  2
me:  2
me:  2
trying: ['CONVERT_TO_CAT', 'AUTO_ADJUST_LEARNING_RATE', 'VARIABLE_FILES', 'SEP_DOLLAR']
me:  2
me:  2
me:  2
me:  2
trying: ['CONVERT_TO_CAT', 'AUTO_ADJUST_LEARNING_RATE', 'VARIABLE_FILES', 'SEP_DOLLAR']
me:  2
me:  2
me:  2
me:  2
trying: ['col']
me:  5
trying: ['Path']
me:  0
trying: ['FASTAI_LEARNING_RATE']
me:  2
trying: ['random']
me:  1
trying: ['shutil']
me: 

In [4]:
%%RecordEvent
%%time
### cell 8 ###

### cell 8 optimized ###

# 1) Identify and set the target column (last convertible to numeric, skipping the first)
target = ''
target_str = ''
for col in df.columns[1:][::-1]:
    try:
        # Coerce to numeric (non-convertible entries become NaN)
        df[col] = pd.to_numeric(df[col], errors='coerce')
        target = col
        target_str = col.replace('/', '-')
        print(f"Target Variable: {target}")
        break
    except Exception:
        continue

# 2) Ensure PARAM_DIR and its config files exist
import os
os.makedirs(PARAM_DIR, exist_ok=True)
for fname in ['cats.txt', 'conts.txt', 'cols_to_delete.txt']:
    open(os.path.join(PARAM_DIR, fname), 'a').close()

# 3) Drop duplicate rows and optionally shuffle
#    (reset index only when shuffling, as in the original)
df = df.drop_duplicates()
if SHUFFLE_DATA:
    df = df.sample(frac=1).reset_index(drop=True)

# 4) Cast boolean columns to uint8
bool_cols = [c for c, dt in df.dtypes.items() if pd.api.types.is_bool_dtype(dt)]
if bool_cols:
    df = df.astype({c: 'uint8' for c in bool_cols})

# 5) Drop any columns listed in cols_to_delete.txt
with open(os.path.join(PARAM_DIR, 'cols_to_delete.txt')) as f:
    cols_to_delete = [line.strip() for line in f if line.strip()]
if cols_to_delete:
    df = df.drop(columns=cols_to_delete, errors='ignore')

# 6) Fill remaining NaNs with 0
df = df.fillna(0)

# 7) Auto-detect categorical vs. continuous
n_unique = df.nunique()
counts   = df.count()
ratios   = n_unique / counts
cats     = ratios[ratios < 0.05].index.tolist()
conts    = ratios[ratios >= 0.05].index.tolist()
# Exclude the target
to_remove = {target}
cats  = [c for c in cats  if c not in to_remove]
conts = [c for c in conts if c not in to_remove]

# 8) Re-convert target to numeric (keep NaNs for non-convertibles)
df[target] = pd.to_numeric(df[target], errors='coerce')

# 9) If VARIABLE_FILES is True, override cats/conts from disk
if VARIABLE_FILES:
    with open(os.path.join(PARAM_DIR, 'cats.txt')) as f:
        cats = [line.strip() for line in f if line.strip()]
    with open(os.path.join(PARAM_DIR, 'conts.txt')) as f:
        conts = [line.strip() for line in f if line.strip()]

# 10) Bulk-convert continuous variables to numeric (leave NaNs)
if conts:
    df[conts] = df[conts].apply(lambda s: pd.to_numeric(s, errors='coerce'))

# 11) If ENABLE_BREAKPOINT, validate each continuous var
if ENABLE_BREAKPOINT:
    cont_list = conts.copy()
    for var in cont_list:
        col = pd.to_numeric(df[var], errors='coerce')
        if col.isna().all():
            print(f"Could not convert {var} to float.")
            cont_list.remove(var)
            if CONVERT_TO_CAT:
                cats.append(var)
        else:
            df[var] = col

Target Variable: Profit_processed


Could not convert City to float.
CPU times: user 428 ms, sys: 96 ms, total: 524 ms
Wall time: 511 ms


In [5]:
%Checkpoint /scratch/jieq/pandax/dias_notebooks/dataranch/supermarket-sales-prediction-xgboost-fastai/src/rewritten/o4_mini_high_small/checkpoints/post_cell_8_try_3.pickle

migration speed (bps): 834515875.8578697
---------------------------
variables to migrate:
target 65
cats 248
pd 72
ratios 638
n_unique 638
SEP_COMMA 28
var 64
df 2474089
f 208
bool_cols 56
fname 67
FASTAI_LEARNING_RATE 24
cols_to_delete 56
REGRESSOR 28
orig_output 446
ENABLE_BREAKPOINT 28
to_remove 216
col 79808
PARAM_DIR 88
random 72
shutil 72
SEP_DOLLAR 24
target_str 65
CONVERT_TO_CAT 24
np 72
AUTO_ADJUST_LEARNING_RATE 24
cont_list 104
counts 638
traceback 72
Path 904
VARIABLE_FILES 24
PROJECT_NAME 59
SHUFFLE_DATA 28
conts 120
warnings 72
benchmark_name 92
factor 28
BENCHMARKS_TO_PATHS 2272
---------------------------
variables to recompute:
[]
---------------------------
cells to recompute:
[]
Checkpoint saved to: /scratch/jieq/pandax/dias_notebooks/dataranch/supermarket-sales-prediction-xgboost-fastai/src/rewritten/o4_mini_high_small/checkpoints/post_cell_8_try_3.pickle


In [6]:
%PrintCellInfo opt_cell_exec_info

======= Cell 0 =======
Input variables ['BENCHMARKS_TO_PATHS', 'Path']
Active variables ['df', 'PARAM_DIR']
Intermediate variables ['factor', 'benchmark_name']
Future variables []
Modified dataframes
======= Cell 1 =======
Input variables ['SEP_COMMA', 'SEP_DOLLAR', 'df']
Active variables ['col', 'df']
Intermediate variables []
Future variables ['PARAM_DIR']
Modified dataframes
  df
    Input columns: set()
    Changed columns: set()
    Created columns: {'Country_processed', 'Profit_processed', 'Postal Code_processed', 'Ship Mode_processed', 'Quantity_processed', 'Sub-Category_processed', 'Category_processed', 'State_processed', 'Sales_processed', 'City_processed', 'Region_processed', 'Segment_processed', 'Discount_processed'}
    Deleted columns: set()
======= Cell 2 =======
Input variables ['df']
Active variables []
Intermediate variables []
Future variables ['col', 'df', 'PARAM_DIR']
Modified dataframes
======= Cell 3 =======
Input variables ['df']
Active variables []
Intermediate 

In [7]:

with open("/scratch/jieq/pandax/dias_notebooks/dataranch/supermarket-sales-prediction-xgboost-fastai/src/opt_cell_exec_info_8_try_3.pkl", "wb") as f:
    pickle.dump(opt_cell_exec_info[8], f)


In [8]:
opt_output = Out.get(4)

In [9]:
df_opt = df
%LoadCheckpoint /scratch/jieq/pandax/dias_notebooks/dataranch/supermarket-sales-prediction-xgboost-fastai/src/small_bench/checkpoints/post_cell_8.pickle
assert compare_df(df_opt, df)

import numpy as np
if os.getenv("USE_GPU") == "True":
    import cudf
from elastic.core.common.pandas import is_type_styler
is_orig_output_pd = isinstance(orig_output, (pd.Series, pd.DataFrame, pd.Index))
is_opt_output_pd = isinstance(opt_output, (pd.Series, pd.DataFrame, pd.Index))
if os.getenv("USE_GPU") == "True":
    is_orig_output_array = isinstance(orig_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray))
    is_opt_output_array = isinstance(opt_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray))
else:
    is_orig_output_array = isinstance(orig_output, np.ndarray)
    is_opt_output_array = isinstance(opt_output, np.ndarray)

is_orig_output_styler = is_type_styler(type(orig_output))
is_opt_output_styler = is_type_styler(type(opt_output))
if is_orig_output_styler and is_opt_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_orig_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_opt_output_styler:
    assert opt_output.to_html() == orig_output

if is_orig_output_pd and is_opt_output_pd:
    assert orig_output.equals(opt_output)
# TODO(jie): this is a hack.
elif ((is_orig_output_pd or is_opt_output_pd) and (is_orig_output_array or is_opt_output_array)) or (is_orig_output_array and is_opt_output_array):
    assert list(orig_output) == list(opt_output)
else:
    assert orig_output == opt_output


trying: ['ENABLE_BREAKPOINT', 'SHUFFLE_DATA', 'REGRESSOR', 'SEP_COMMA']
me:  2
me:  2
me:  2
me:  2
trying: ['ENABLE_BREAKPOINT', 'SHUFFLE_DATA', 'REGRESSOR', 'SEP_COMMA']
me:  2
me:  2
me:  2
me:  2
trying: ['ENABLE_BREAKPOINT', 'SHUFFLE_DATA', 'REGRESSOR', 'SEP_COMMA']
me:  2
me:  2
me:  2
me:  2
trying: ['ENABLE_BREAKPOINT', 'SHUFFLE_DATA', 'REGRESSOR', 'SEP_COMMA']
me:  2
me:  2
me:  2
me:  2
trying: ['focus_cont', 'var', 'cont', 'n']
me:  19
me:  19
me:  19
me:  19
trying: ['focus_cont', 'var', 'cont', 'n']
me:  19
me:  19
me:  19
me:  19
trying: ['target', 'target_str']
me:  19
me:  19
trying: ['target', 'target_str']
me:  19
me:  19
trying: ['AUTO_ADJUST_LEARNING_RATE', 'VARIABLE_FILES', 'SEP_DOLLAR', 'CONVERT_TO_CAT']
me:  2
me:  2
me:  2
me:  2
trying: ['AUTO_ADJUST_LEARNING_RATE', 'VARIABLE_FILES', 'SEP_DOLLAR', 'CONVERT_TO_CAT']
me:  2
me:  2
me:  2
me:  2
trying: ['factor', 'i']
me:  3
me:  19
trying: ['factor', 'i']
me:  3
me:  19
trying: ['AUTO_ADJUST_LEARNING_RATE', 'VAR

ValueError: Shape mismatch: (9976, 26) vs (9960, 26)